# Clean Model Lifecycle — Precheck

This notebook compares selected classical and ensemble-learning models using
the same leakage-safe group split.

No models are registered or deployed during this precheck. The results will
determine the clean Champion–Challenger sequence used in the final workflow.

In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
df = spark.table(
    "default.refined_mtc_dataset"
).toPandas()

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["target"].value_counts().sort_index())

Dataset shape: (1107, 197)

Target distribution:
target
0    814
1    293
Name: count, dtype: int64


In [0]:
df = spark.table(
    "default.refined_mtc_dataset"
).toPandas()

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["target"].value_counts().sort_index())

Dataset shape: (1107, 197)

Target distribution:
target
0    814
1    293
Name: count, dtype: int64


In [0]:
identifier_columns = [
    "dataset_folder",
    "mtc_file",
    "label_file",
    "segment_id",
    "start",
    "end",
    "raw_label"
]

columns_to_remove = [
    column
    for column in identifier_columns
    if column in df.columns
]

numeric_df = df.select_dtypes(
    include=[np.number]
).copy()

feature_columns = [
    column
    for column in numeric_df.columns
    if column not in columns_to_remove + ["target"]
]

X = numeric_df[feature_columns].copy()

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

y = df["target"].astype(int).copy()

groups = (
    df["dataset_folder"]
    .astype(str)
    .copy()
)

print("Feature matrix:", X.shape)
print("Target shape:", y.shape)
print("Number of experiments:", groups.nunique())
print("Target values:", sorted(y.unique()))

Feature matrix: (1107, 190)
Target shape: (1107,)
Number of experiments: 24
Target values: [np.int64(0), np.int64(1)]


In [0]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.80,
        colsample_bytree=0.80,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    )
}

print("Candidate models:", list(models.keys()))

Candidate models: ['Random Forest', 'Extra Trees', 'Gradient Boosting', 'XGBoost', 'Decision Tree']


In [0]:
def evaluate_model(
    model_name,
    classifier,
    X_train,
    y_train,
    X_test,
    y_test
):
    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "classifier",
            classifier
        )
    ])

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    probabilities = pipeline.predict_proba(
        X_test
    )[:, 1]

    metrics = {
        "Model": model_name,

        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "F1 Score": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        ),

        "True Negatives": int(
            confusion_matrix(
                y_test,
                predictions
            )[0, 0]
        ),

        "False Positives": int(
            confusion_matrix(
                y_test,
                predictions
            )[0, 1]
        ),

        "False Negatives": int(
            confusion_matrix(
                y_test,
                predictions
            )[1, 0]
        ),

        "True Positives": int(
            confusion_matrix(
                y_test,
                predictions
            )[1, 1]
        )
    }

    return pipeline, metrics

In [0]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    group_splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

train_groups = groups.iloc[train_index]
test_groups = groups.iloc[test_index]

experiment_overlap = set(
    train_groups.unique()
).intersection(
    set(test_groups.unique())
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Experiment overlap:", experiment_overlap)

assert len(experiment_overlap) == 0

print("Leakage-safe group split confirmed.")

Training rows: 942
Testing rows: 165
Experiment overlap: set()
Leakage-safe group split confirmed.


In [0]:
trained_models = {}
model_results = []

for model_name, classifier in models.items():

    print(
        f"Training {model_name}..."
    )

    trained_pipeline, metrics = evaluate_model(
        model_name=model_name,
        classifier=classifier,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )

    trained_models[model_name] = (
        trained_pipeline
    )

    model_results.append(metrics)

    print(
        f"{model_name} completed:",
        f"F1={metrics['F1 Score']:.4f},",
        f"Recall={metrics['Recall']:.4f}"
    )

Training Random Forest...
Random Forest completed: F1=0.8224, Recall=0.9167
Training Extra Trees...
Extra Trees completed: F1=0.8515, Recall=0.8958
Training Gradient Boosting...
Gradient Boosting completed: F1=0.8505, Recall=0.9479
Training XGBoost...
XGBoost completed: F1=0.8311, Recall=0.9479
Training Decision Tree...
Decision Tree completed: F1=0.8867, Recall=0.9375


In [0]:
results_df = pd.DataFrame(
    model_results
)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC"
]

results_df[metric_columns] = (
    results_df[metric_columns]
    .round(4)
)

display(results_df)

Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,True Negatives,False Positives,False Negatives,True Positives
Decision Tree,0.8606,0.8411,0.9375,0.8867,0.8463,52,17,6,90
Extra Trees,0.8182,0.8113,0.8958,0.8515,0.9404,49,20,10,86
Gradient Boosting,0.8061,0.7712,0.9479,0.8505,0.939,42,27,5,91
XGBoost,0.7758,0.7398,0.9479,0.8311,0.8915,37,32,5,91
Random Forest,0.7697,0.7458,0.9167,0.8224,0.9004,39,30,8,88


In [0]:
print("MODEL RANKING BY F1 SCORE")
print("-------------------------")

print(
    results_df[
        [
            "Model",
            "F1 Score",
            "Recall",
            "Precision",
            "ROC-AUC"
        ]
    ].to_string(index=False)
)

MODEL RANKING BY F1 SCORE
-------------------------
            Model  F1 Score  Recall  Precision  ROC-AUC
    Decision Tree    0.8867  0.9375     0.8411   0.8463
      Extra Trees    0.8515  0.8958     0.8113   0.9404
Gradient Boosting    0.8505  0.9479     0.7712   0.9390
          XGBoost    0.8311  0.9479     0.7398   0.8915
    Random Forest    0.8224  0.9167     0.7458   0.9004


## Production Lifecycle Plan

The promotion rule requires:

- an F1-score improvement of at least `0.01`;
- anomaly recall must not decrease by more than `0.02`.

Models that fail this gate remain experimental runs and are not registered as
production versions.

In [0]:
MINIMUM_F1_GAIN = 0.01
RECALL_TOLERANCE = 0.02


def get_metrics(model_name):
    row = results_df[
        results_df["Model"] == model_name
    ].iloc[0]

    return {
        "f1_score": float(row["F1 Score"]),
        "recall": float(row["Recall"]),
        "precision": float(row["Precision"]),
        "roc_auc": float(row["ROC-AUC"])
    }


def evaluate_promotion(
    champion_name,
    challenger_name
):
    champion = get_metrics(champion_name)
    challenger = get_metrics(challenger_name)

    f1_gain = (
        challenger["f1_score"]
        - champion["f1_score"]
    )

    recall_change = (
        challenger["recall"]
        - champion["recall"]
    )

    f1_passed = (
        f1_gain >= MINIMUM_F1_GAIN
    )

    recall_passed = (
        challenger["recall"]
        >= champion["recall"]
        - RECALL_TOLERANCE
    )

    promote = (
        f1_passed
        and recall_passed
    )

    return {
        "Champion": champion_name,
        "Challenger": challenger_name,
        "Champion F1": champion["f1_score"],
        "Challenger F1": challenger["f1_score"],
        "F1 Gain": round(f1_gain, 4),
        "Champion Recall": champion["recall"],
        "Challenger Recall": challenger["recall"],
        "Recall Change": round(recall_change, 4),
        "F1 Gate": f1_passed,
        "Recall Gate": recall_passed,
        "Promote": promote
    }

In [0]:
promotion_tests = []

# Current production model begins with Random Forest
current_champion = "Random Forest"

# Challenger 1: XGBoost
xgb_decision = evaluate_promotion(
    current_champion,
    "XGBoost"
)

promotion_tests.append(xgb_decision)

if xgb_decision["Promote"]:
    current_champion = "XGBoost"


# Challenger 2: Gradient Boosting
gb_decision = evaluate_promotion(
    current_champion,
    "Gradient Boosting"
)

promotion_tests.append(gb_decision)

if gb_decision["Promote"]:
    current_champion = "Gradient Boosting"


# Challenger 3: Extra Trees
extra_decision = evaluate_promotion(
    current_champion,
    "Extra Trees"
)

promotion_tests.append(extra_decision)

if extra_decision["Promote"]:
    current_champion = "Extra Trees"


# Final challenger: Decision Tree
dt_decision = evaluate_promotion(
    current_champion,
    "Decision Tree"
)

promotion_tests.append(dt_decision)

if dt_decision["Promote"]:
    current_champion = "Decision Tree"


promotion_df = pd.DataFrame(
    promotion_tests
)

display(promotion_df)

print("Final selected Champion:", current_champion)

Champion,Challenger,Champion F1,Challenger F1,F1 Gain,Champion Recall,Challenger Recall,Recall Change,F1 Gate,Recall Gate,Promote
Random Forest,XGBoost,0.8224,0.8311,0.0087,0.9167,0.9479,0.0312,false,true,false
Random Forest,Gradient Boosting,0.8224,0.8505,0.0281,0.9167,0.9479,0.0312,true,true,true
Gradient Boosting,Extra Trees,0.8505,0.8515,0.001,0.9479,0.8958,-0.0521,false,false,false
Gradient Boosting,Decision Tree,0.8505,0.8867,0.0362,0.9479,0.9375,-0.0104,true,true,true


Final selected Champion: Decision Tree


In [0]:
lifecycle_plan = pd.DataFrame([
    {
        "Order": 1,
        "Model": "Random Forest",
        "Role": "Initial production model",
        "Expected registry result": "Version 1"
    },
    {
        "Order": 2,
        "Model": "XGBoost",
        "Role": "First challenger",
        "Expected registry result": "Rejected — not registered"
    },
    {
        "Order": 3,
        "Model": "Gradient Boosting",
        "Role": "Successful challenger",
        "Expected registry result": "Version 2"
    },
    {
        "Order": 4,
        "Model": "Extra Trees",
        "Role": "Recall-sensitive challenger",
        "Expected registry result": "Rejected — not registered"
    },
    {
        "Order": 5,
        "Model": "Decision Tree",
        "Role": "Final improved challenger",
        "Expected registry result": "Version 3"
    }
])

display(lifecycle_plan)

Order,Model,Role,Expected registry result
1,Random Forest,Initial production model,Version 1
2,XGBoost,First challenger,Rejected — not registered
3,Gradient Boosting,Successful challenger,Version 2
4,Extra Trees,Recall-sensitive challenger,Rejected — not registered
5,Decision Tree,Final improved challenger,Version 3


In [0]:
lifecycle_plan = pd.DataFrame([
    {
        "Order": 1,
        "Model": "Random Forest",
        "Role": "Initial production model",
        "Expected registry result": "Version 1"
    },
    {
        "Order": 2,
        "Model": "XGBoost",
        "Role": "First challenger",
        "Expected registry result": "Rejected — not registered"
    },
    {
        "Order": 3,
        "Model": "Gradient Boosting",
        "Role": "Successful challenger",
        "Expected registry result": "Version 2"
    },
    {
        "Order": 4,
        "Model": "Extra Trees",
        "Role": "Recall-sensitive challenger",
        "Expected registry result": "Rejected — not registered"
    },
    {
        "Order": 5,
        "Model": "Decision Tree",
        "Role": "Final improved challenger",
        "Expected registry result": "Version 3"
    }
])

display(lifecycle_plan)

Order,Model,Role,Expected registry result
1,Random Forest,Initial production model,Version 1
2,XGBoost,First challenger,Rejected — not registered
3,Gradient Boosting,Successful challenger,Version 2
4,Extra Trees,Recall-sensitive challenger,Rejected — not registered
5,Decision Tree,Final improved challenger,Version 3


In [0]:
def log_candidate_to_mlflow(
    model_name,
    model,
    role,
    promotion_result=None
):
    metrics = get_metrics(
        model_name
    )

    safe_run_name = (
        model_name
        .lower()
        .replace(" ", "_")
    )

    with mlflow.start_run(
        run_name=safe_run_name
    ) as run:

        mlflow.log_param(
            "model_name",
            model_name
        )

        mlflow.log_param(
            "lifecycle_role",
            role
        )

        mlflow.log_param(
            "split_method",
            "GroupShuffleSplit"
        )

        mlflow.log_param(
            "random_state",
            42
        )

        mlflow.log_param(
            "training_rows",
            len(X_train)
        )

        mlflow.log_param(
            "testing_rows",
            len(X_test)
        )

        mlflow.log_param(
            "feature_count",
            X_train.shape[1]
        )

        if promotion_result is not None:
            mlflow.log_param(
                "promotion_passed",
                bool(
                    promotion_result
                )
            )

        mlflow.log_metrics({
            "accuracy": float(
                results_df.loc[
                    results_df["Model"]
                    == model_name,
                    "Accuracy"
                ].iloc[0]
            ),
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1_score"],
            "roc_auc": metrics["roc_auc"]
        })

        signature = infer_signature(
            X_train,
            model.predict(X_train)
        )

        model_info = (
            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="model",
                signature=signature,
                input_example=X_train.head(3)
            )
        )

        return {
            "run_id": run.info.run_id,
            "model_uri": model_info.model_uri,
            "metrics": metrics
        }

In [0]:
import mlflow
import mlflow.sklearn

from mlflow import MlflowClient
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

EXPERIMENT_NAME = "/Shared/cnc_milling_clean_lifecycle"
MODEL_NAME = "workspace.default.cnc_milling_anomaly_final"

mlflow.set_experiment(EXPERIMENT_NAME)

registry_client = MlflowClient()

print("MLflow version:", mlflow.__version__)
print("Experiment:", EXPERIMENT_NAME)
print("Registered model:", MODEL_NAME)

2026/07/26 18:07:43 INFO mlflow.tracking.fluent: Experiment with name '/Shared/cnc_milling_clean_lifecycle' does not exist. Creating a new experiment.


MLflow version: 3.8.1
Experiment: /Shared/cnc_milling_clean_lifecycle
Registered model: workspace.default.cnc_milling_anomaly_final


In [0]:
random_forest_log = log_candidate_to_mlflow(
    model_name="Random Forest",
    model=trained_models["Random Forest"],
    role="initial_production_model",
    promotion_result=True
)

print(
    "Random Forest run ID:",
    random_forest_log["run_id"]
)

print(
    "Random Forest model URI:",
    random_forest_log["model_uri"]
)

2026/07/26 18:07:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/197268204771846/models/m-cb4d5373e9ab408f9501d682a7e94946?o=7474646333398352


Random Forest run ID: e244fbe8adfc4ab18f171dadbf99fbe9
Random Forest model URI: models:/m-cb4d5373e9ab408f9501d682a7e94946


In [0]:
registered_v1 = mlflow.register_model(
    model_uri=random_forest_log["model_uri"],
    name=MODEL_NAME
)

VERSION_1 = int(registered_v1.version)

print("Registered Version:", VERSION_1)
print("Model: Random Forest")

Successfully registered model 'workspace.default.cnc_milling_anomaly_final'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered Version: 1
Model: Random Forest


🔗 Created version '1' of model 'workspace.default.cnc_milling_anomaly_final': https://dbc-700d842f-f782.cloud.databricks.com/explore/data/models/workspace/default/cnc_milling_anomaly_final/version/1?o=7474646333398352


In [0]:
registry_client = MlflowClient()

registry_client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=VERSION_1
)

champion = registry_client.get_model_version_by_alias(
    name=MODEL_NAME,
    alias="Champion"
)

print("INITIAL PRODUCTION STATE")
print("------------------------")
print("Champion model: Random Forest")
print("Champion version:", champion.version)
print(
    "F1 score:",
    random_forest_log["metrics"]["f1_score"]
)
print(
    "Recall:",
    random_forest_log["metrics"]["recall"]
)

INITIAL PRODUCTION STATE
------------------------
Champion model: Random Forest
Champion version: 1
F1 score: 0.8224
Recall: 0.9167


In [0]:
import time
from mlflow.deployments import get_deploy_client

ENDPOINT_NAME = "cnc-milling-final-endpoint"

deployment_client = get_deploy_client(
    "databricks"
)

served_entity_v1 = "random-forest-v1"

endpoint_v1 = deployment_client.create_endpoint(
    name=ENDPOINT_NAME,
    config={
        "served_entities": [
            {
                "name": served_entity_v1,
                "entity_name": MODEL_NAME,
                "entity_version": str(VERSION_1),
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": served_entity_v1,
                    "traffic_percentage": 100
                }
            ]
        }
    }
)

print("Endpoint creation requested.")
print("Endpoint:", ENDPOINT_NAME)
print("Model:", MODEL_NAME)
print("Version:", VERSION_1)

Endpoint creation requested.
Endpoint: cnc-milling-final-endpoint
Model: workspace.default.cnc_milling_anomaly_final
Version: 1


In [0]:
def wait_for_endpoint(
    client,
    endpoint_name,
    maximum_minutes=25
):
    total_attempts = maximum_minutes * 3

    for attempt in range(total_attempts):
        endpoint_info = client.get_endpoint(
            endpoint=endpoint_name
        )

        state = endpoint_info.get(
            "state",
            {}
        )

        ready_state = state.get(
            "ready"
        )

        update_state = (
            state.get("config_update")
            or state.get("update_state")
        )

        print(
            f"Attempt {attempt + 1}:",
            f"ready={ready_state},",
            f"update={update_state}"
        )

        if (
            ready_state == "READY"
            and update_state in [
                None,
                "NOT_UPDATING"
            ]
        ):
            print(
                "\nEndpoint is ready."
            )

            return endpoint_info

        time.sleep(20)

    raise TimeoutError(
        "Endpoint did not become ready "
        "within 25 minutes."
    )

In [0]:
endpoint_v1_info = wait_for_endpoint(
    deployment_client,
    ENDPOINT_NAME
)

Attempt 1: ready=NOT_READY, update=IN_PROGRESS
Attempt 2: ready=NOT_READY, update=IN_PROGRESS
Attempt 3: ready=NOT_READY, update=IN_PROGRESS
Attempt 4: ready=NOT_READY, update=IN_PROGRESS
Attempt 5: ready=NOT_READY, update=IN_PROGRESS
Attempt 6: ready=NOT_READY, update=IN_PROGRESS
Attempt 7: ready=NOT_READY, update=IN_PROGRESS
Attempt 8: ready=NOT_READY, update=IN_PROGRESS
Attempt 9: ready=NOT_READY, update=IN_PROGRESS
Attempt 10: ready=NOT_READY, update=IN_PROGRESS
Attempt 11: ready=NOT_READY, update=IN_PROGRESS
Attempt 12: ready=NOT_READY, update=IN_PROGRESS
Attempt 13: ready=NOT_READY, update=IN_PROGRESS
Attempt 14: ready=NOT_READY, update=IN_PROGRESS
Attempt 15: ready=NOT_READY, update=IN_PROGRESS
Attempt 16: ready=NOT_READY, update=IN_PROGRESS
Attempt 17: ready=NOT_READY, update=IN_PROGRESS
Attempt 18: ready=NOT_READY, update=IN_PROGRESS
Attempt 19: ready=NOT_READY, update=IN_PROGRESS
Attempt 20: ready=NOT_READY, update=IN_PROGRESS
Attempt 21: ready=NOT_READY, update=IN_PROGRESS
A

In [0]:
endpoint_v1_info = deployment_client.get_endpoint(
    endpoint=ENDPOINT_NAME
)

served_v1 = (
    endpoint_v1_info["config"]
    ["served_entities"][0]
)

champion = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

served_version = str(
    served_v1["entity_version"]
)

champion_version = str(
    champion.version
)

versions_match = (
    served_version == champion_version
)

print("VERSION 1 DEPLOYMENT COMPLETE")
print("-----------------------------")
print("Endpoint:", ENDPOINT_NAME)
print("Served model:", served_v1["entity_name"])
print("Served version:", served_version)
print("Champion version:", champion_version)
print(
    "Champion and endpoint match:",
    versions_match
)

VERSION 1 DEPLOYMENT COMPLETE
-----------------------------
Endpoint: cnc-milling-final-endpoint
Served model: workspace.default.cnc_milling_anomaly_final
Served version: 1
Champion version: 1
Champion and endpoint match: True


In [0]:
def create_endpoint_payload(input_df):
    clean_df = (
        input_df
        .astype(object)
        .where(pd.notna(input_df), None)
    )

    return {
        "dataframe_split": {
            "columns": clean_df.columns.tolist(),
            "data": clean_df.values.tolist()
        }
    }

In [0]:
normal_positions = np.where(
    y_test.to_numpy() == 0
)[0]

normal_position = int(
    normal_positions[0]
)

version_1_sample = X_test.iloc[
    [normal_position]
].copy()

actual_label_v1 = int(
    y_test.iloc[normal_position]
)

version_1_payload = create_endpoint_payload(
    version_1_sample
)

version_1_response = deployment_client.predict(
    endpoint=ENDPOINT_NAME,
    inputs=version_1_payload
)

print("VERSION 1 ENDPOINT TEST")
print("-----------------------")
print("Actual label:", actual_label_v1)
print("Endpoint response:", version_1_response)

VERSION 1 ENDPOINT TEST
-----------------------
Actual label: 0
Endpoint response: {'predictions': [1]}


In [0]:
local_prediction_v1 = int(
    trained_models["Random Forest"]
    .predict(version_1_sample)[0]
)

endpoint_prediction_v1 = int(
    version_1_response["predictions"][0]
)

print("VERSION 1 CONSISTENCY CHECK")
print("---------------------------")
print("Actual label:", actual_label_v1)
print("Local model prediction:", local_prediction_v1)
print("Endpoint prediction:", endpoint_prediction_v1)
print(
    "Local and endpoint match:",
    local_prediction_v1 == endpoint_prediction_v1
)

VERSION 1 CONSISTENCY CHECK
---------------------------
Actual label: 0
Local model prediction: 1
Endpoint prediction: 1
Local and endpoint match: True


In [0]:
all_local_predictions_v1 = (
    trained_models["Random Forest"]
    .predict(X_test)
)

correct_normal_positions = np.where(
    (y_test.to_numpy() == 0)
    & (all_local_predictions_v1 == 0)
)[0]

print(
    "Correctly predicted normal samples:",
    len(correct_normal_positions)
)

normal_position = int(
    correct_normal_positions[0]
)

correct_normal_sample_v1 = X_test.iloc[
    [normal_position]
].copy()

correct_normal_actual_v1 = int(
    y_test.iloc[normal_position]
)

correct_normal_local_prediction_v1 = int(
    trained_models["Random Forest"]
    .predict(correct_normal_sample_v1)[0]
)

print("Actual label:", correct_normal_actual_v1)
print(
    "Local prediction:",
    correct_normal_local_prediction_v1
)

Correctly predicted normal samples: 39
Actual label: 0
Local prediction: 0


In [0]:
correct_normal_payload_v1 = (
    create_endpoint_payload(
        correct_normal_sample_v1
    )
)

correct_normal_response_v1 = (
    deployment_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs=correct_normal_payload_v1
    )
)

correct_normal_endpoint_prediction_v1 = int(
    correct_normal_response_v1[
        "predictions"
    ][0]
)

print("VERSION 1 NORMAL INFERENCE")
print("--------------------------")
print("Actual label:", correct_normal_actual_v1)

print(
    "Local prediction:",
    correct_normal_local_prediction_v1
)

print(
    "Endpoint prediction:",
    correct_normal_endpoint_prediction_v1
)

print(
    "Serving consistency:",
    correct_normal_local_prediction_v1
    == correct_normal_endpoint_prediction_v1
)

print(
    "Correct prediction:",
    correct_normal_actual_v1
    == correct_normal_endpoint_prediction_v1
)

VERSION 1 NORMAL INFERENCE
--------------------------
Actual label: 0
Local prediction: 0
Endpoint prediction: 0
Serving consistency: True
Correct prediction: True


In [0]:
xgb_decision = evaluate_promotion(
    champion_name="Random Forest",
    challenger_name="XGBoost"
)

print("XGBOOST PROMOTION CHECK")
print("-----------------------")

for key, value in xgb_decision.items():
    print(f"{key}: {value}")

XGBOOST PROMOTION CHECK
-----------------------
Champion: Random Forest
Challenger: XGBoost
Champion F1: 0.8224
Challenger F1: 0.8311
F1 Gain: 0.0087
Champion Recall: 0.9167
Challenger Recall: 0.9479
Recall Change: 0.0312
F1 Gate: False
Recall Gate: True
Promote: False


In [0]:
xgboost_log = log_candidate_to_mlflow(
    model_name="XGBoost",
    model=trained_models["XGBoost"],
    role="rejected_challenger",
    promotion_result=xgb_decision["Promote"]
)

print("XGBoost run ID:", xgboost_log["run_id"])
print("Promotion passed:", xgb_decision["Promote"])
print("Registry action: Not registered")

2026/07/26 18:30:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/197268204771846/models/m-206e59185e4b4be19d981a201bf7c43f?o=7474646333398352


XGBoost run ID: e83c4d31f5494653b6a54d8982178513
Promotion passed: False
Registry action: Not registered


In [0]:
champion_after_xgb = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

endpoint_after_xgb = deployment_client.get_endpoint(
    endpoint=ENDPOINT_NAME
)

served_after_xgb = (
    endpoint_after_xgb["config"]
    ["served_entities"][0]
)

production_unchanged = (
    str(champion_after_xgb.version) == "1"
    and str(served_after_xgb["entity_version"]) == "1"
)

print("PRODUCTION STATE AFTER XGBOOST")
print("------------------------------")
print("XGBoost promotion:", xgb_decision["Promote"])
print("Champion version:", champion_after_xgb.version)
print(
    "Endpoint version:",
    served_after_xgb["entity_version"]
)
print(
    "Production model unchanged:",
    production_unchanged
)

PRODUCTION STATE AFTER XGBOOST
------------------------------
XGBoost promotion: False
Champion version: 1
Endpoint version: 1
Production model unchanged: True


In [0]:
gb_decision = evaluate_promotion(
    champion_name="Random Forest",
    challenger_name="Gradient Boosting"
)

print("GRADIENT BOOSTING PROMOTION CHECK")
print("---------------------------------")

for key, value in gb_decision.items():
    print(f"{key}: {value}")

assert gb_decision["Promote"] is True

print("\nQuality gate passed successfully.")

GRADIENT BOOSTING PROMOTION CHECK
---------------------------------
Champion: Random Forest
Challenger: Gradient Boosting
Champion F1: 0.8224
Challenger F1: 0.8505
F1 Gain: 0.0281
Champion Recall: 0.9167
Challenger Recall: 0.9479
Recall Change: 0.0312
F1 Gate: True
Recall Gate: True
Promote: True

Quality gate passed successfully.


In [0]:
gradient_boosting_log = log_candidate_to_mlflow(
    model_name="Gradient Boosting",
    model=trained_models["Gradient Boosting"],
    role="successful_challenger",
    promotion_result=gb_decision["Promote"]
)

print(
    "Gradient Boosting run ID:",
    gradient_boosting_log["run_id"]
)

print(
    "Promotion passed:",
    gb_decision["Promote"]
)

2026/07/26 18:32:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/197268204771846/models/m-2d126596313f4c0db31cd88b92c5a5cb?o=7474646333398352


Gradient Boosting run ID: a15c691b95304e2db40e38d592a304cb
Promotion passed: True


In [0]:
registered_v2 = mlflow.register_model(
    model_uri=gradient_boosting_log["model_uri"],
    name=MODEL_NAME
)

VERSION_2 = int(registered_v2.version)

print("REGISTERED SUCCESSFUL CHALLENGER")
print("--------------------------------")
print("Model: Gradient Boosting")
print("Registered version:", VERSION_2)

Registered model 'workspace.default.cnc_milling_anomaly_final' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.cnc_milling_anomaly_final': https://dbc-700d842f-f782.cloud.databricks.com/explore/data/models/workspace/default/cnc_milling_anomaly_final/version/2?o=7474646333398352


REGISTERED SUCCESSFUL CHALLENGER
--------------------------------
Model: Gradient Boosting
Registered version: 2


In [0]:
if not gb_decision["Promote"]:
    raise ValueError(
        "Gradient Boosting failed the quality gate. "
        "Production was not changed."
    )

served_entity_v2 = (
    f"gradient-boosting-v{VERSION_2}"
)

deployment_client.update_endpoint_config(
    endpoint=ENDPOINT_NAME,
    config={
        "served_entities": [
            {
                "name": served_entity_v2,
                "entity_name": MODEL_NAME,
                "entity_version": str(VERSION_2),
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": served_entity_v2,
                    "traffic_percentage": 100
                }
            ]
        }
    }
)

print("AUTOMATIC DEPLOYMENT REQUESTED")
print("------------------------------")
print("Previous production version: 1")
print("New candidate version:", VERSION_2)
print("Model: Gradient Boosting")

AUTOMATIC DEPLOYMENT REQUESTED
------------------------------
Previous production version: 1
New candidate version: 2
Model: Gradient Boosting


In [0]:
endpoint_v2_info = wait_for_endpoint(
    deployment_client,
    ENDPOINT_NAME
)

Attempt 1: ready=READY, update=IN_PROGRESS
Attempt 2: ready=READY, update=IN_PROGRESS
Attempt 3: ready=READY, update=IN_PROGRESS
Attempt 4: ready=READY, update=IN_PROGRESS
Attempt 5: ready=READY, update=IN_PROGRESS
Attempt 6: ready=READY, update=IN_PROGRESS
Attempt 7: ready=READY, update=IN_PROGRESS
Attempt 8: ready=READY, update=IN_PROGRESS
Attempt 9: ready=READY, update=IN_PROGRESS
Attempt 10: ready=READY, update=IN_PROGRESS
Attempt 11: ready=READY, update=IN_PROGRESS
Attempt 12: ready=READY, update=IN_PROGRESS
Attempt 13: ready=READY, update=IN_PROGRESS
Attempt 14: ready=READY, update=IN_PROGRESS
Attempt 15: ready=READY, update=IN_PROGRESS
Attempt 16: ready=READY, update=IN_PROGRESS
Attempt 17: ready=READY, update=IN_PROGRESS
Attempt 18: ready=READY, update=IN_PROGRESS
Attempt 19: ready=READY, update=IN_PROGRESS
Attempt 20: ready=READY, update=IN_PROGRESS
Attempt 21: ready=READY, update=IN_PROGRESS
Attempt 22: ready=READY, update=IN_PROGRESS
Attempt 23: ready=READY, update=IN_PROGRE

In [0]:
registry_client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=VERSION_2
)

champion_v2 = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

print("Champion alias moved successfully.")
print("Champion version:", champion_v2.version)

Champion alias moved successfully.
Champion version: 2


In [0]:
endpoint_v2_info = deployment_client.get_endpoint(
    endpoint=ENDPOINT_NAME
)

served_v2 = (
    endpoint_v2_info["config"]
    ["served_entities"][0]
)

champion_v2 = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

champion_endpoint_match_v2 = (
    str(champion_v2.version)
    == str(served_v2["entity_version"])
)

print("FIRST AUTOMATIC MODEL REPLACEMENT")
print("---------------------------------")
print("Previous model: Random Forest")
print("Previous version: 1")
print("New model: Gradient Boosting")
print("New Champion version:", champion_v2.version)
print(
    "Endpoint version:",
    served_v2["entity_version"]
)
print(
    "Champion and endpoint match:",
    champion_endpoint_match_v2
)
print(
    "Automatic replacement successful:",
    champion_endpoint_match_v2
    and str(champion_v2.version) == str(VERSION_2)
)

FIRST AUTOMATIC MODEL REPLACEMENT
---------------------------------
Previous model: Random Forest
Previous version: 1
New model: Gradient Boosting
New Champion version: 2
Endpoint version: 2
Champion and endpoint match: True
Automatic replacement successful: True


In [0]:
extra_decision = evaluate_promotion(
    champion_name="Gradient Boosting",
    challenger_name="Extra Trees"
)

print("EXTRA TREES PROMOTION CHECK")
print("---------------------------")

for key, value in extra_decision.items():
    print(f"{key}: {value}")

EXTRA TREES PROMOTION CHECK
---------------------------
Champion: Gradient Boosting
Challenger: Extra Trees
Champion F1: 0.8505
Challenger F1: 0.8515
F1 Gain: 0.001
Champion Recall: 0.9479
Challenger Recall: 0.8958
Recall Change: -0.0521
F1 Gate: False
Recall Gate: False
Promote: False


In [0]:
extra_trees_log = log_candidate_to_mlflow(
    model_name="Extra Trees",
    model=trained_models["Extra Trees"],
    role="rejected_challenger",
    promotion_result=extra_decision["Promote"]
)

print("Extra Trees run ID:", extra_trees_log["run_id"])
print("Promotion passed:", extra_decision["Promote"])
print("Registry action: Not registered")

2026/07/26 18:45:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/197268204771846/models/m-e2aeebc8ec684efbaec80688dbffd50d?o=7474646333398352


Extra Trees run ID: 3abff56ada1c4c4c95d60de579771de0
Promotion passed: False
Registry action: Not registered


In [0]:
champion_after_extra = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

endpoint_after_extra = (
    deployment_client
    .get_endpoint(
        endpoint=ENDPOINT_NAME
    )
)

served_after_extra = (
    endpoint_after_extra["config"]
    ["served_entities"][0]
)

production_unchanged_after_extra = (
    str(champion_after_extra.version) == str(VERSION_2)
    and
    str(served_after_extra["entity_version"]) == str(VERSION_2)
)

print("PRODUCTION STATE AFTER EXTRA TREES")
print("----------------------------------")
print("Extra Trees promotion:", extra_decision["Promote"])
print("Champion version:", champion_after_extra.version)
print(
    "Endpoint version:",
    served_after_extra["entity_version"]
)
print(
    "Production model unchanged:",
    production_unchanged_after_extra
)

PRODUCTION STATE AFTER EXTRA TREES
----------------------------------
Extra Trees promotion: False
Champion version: 2
Endpoint version: 2
Production model unchanged: True


In [0]:
dt_decision = evaluate_promotion(
    champion_name="Gradient Boosting",
    challenger_name="Decision Tree"
)

print("DECISION TREE PROMOTION CHECK")
print("-----------------------------")

for key, value in dt_decision.items():
    print(f"{key}: {value}")

assert bool(dt_decision["Promote"]) is True

print("\nDecision Tree passed the quality gate.")

DECISION TREE PROMOTION CHECK
-----------------------------
Champion: Gradient Boosting
Challenger: Decision Tree
Champion F1: 0.8505
Challenger F1: 0.8867
F1 Gain: 0.0362
Champion Recall: 0.9479
Challenger Recall: 0.9375
Recall Change: -0.0104
F1 Gate: True
Recall Gate: True
Promote: True

Decision Tree passed the quality gate.


In [0]:
decision_tree_log = log_candidate_to_mlflow(
    model_name="Decision Tree",
    model=trained_models["Decision Tree"],
    role="final_successful_challenger",
    promotion_result=dt_decision["Promote"]
)

print(
    "Decision Tree run ID:",
    decision_tree_log["run_id"]
)

print(
    "Promotion passed:",
    dt_decision["Promote"]
)

2026/07/26 18:47:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/197268204771846/models/m-133a3a62f1b14480b412c53e127fd310?o=7474646333398352


Decision Tree run ID: e0fd3391a6a749bcbd58cf7de50c0306
Promotion passed: True


In [0]:
registered_v3 = mlflow.register_model(
    model_uri=decision_tree_log["model_uri"],
    name=MODEL_NAME
)

VERSION_3 = int(registered_v3.version)

print("FINAL CHALLENGER REGISTERED")
print("---------------------------")
print("Model: Decision Tree")
print("Registered version:", VERSION_3)

Registered model 'workspace.default.cnc_milling_anomaly_final' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.default.cnc_milling_anomaly_final': https://dbc-700d842f-f782.cloud.databricks.com/explore/data/models/workspace/default/cnc_milling_anomaly_final/version/3?o=7474646333398352


FINAL CHALLENGER REGISTERED
---------------------------
Model: Decision Tree
Registered version: 3


In [0]:
if not bool(dt_decision["Promote"]):
    raise ValueError(
        "Decision Tree failed the quality gate. "
        "Production was not changed."
    )

served_entity_v3 = (
    f"decision-tree-v{VERSION_3}"
)

deployment_client.update_endpoint_config(
    endpoint=ENDPOINT_NAME,
    config={
        "served_entities": [
            {
                "name": served_entity_v3,
                "entity_name": MODEL_NAME,
                "entity_version": str(VERSION_3),
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": served_entity_v3,
                    "traffic_percentage": 100
                }
            ]
        }
    }
)

print("FINAL AUTOMATIC DEPLOYMENT REQUESTED")
print("------------------------------------")
print("Previous Champion version:", VERSION_2)
print("New candidate version:", VERSION_3)
print("New model: Decision Tree")

FINAL AUTOMATIC DEPLOYMENT REQUESTED
------------------------------------
Previous Champion version: 2
New candidate version: 3
New model: Decision Tree


In [0]:
endpoint_v3_info = wait_for_endpoint(
    deployment_client,
    ENDPOINT_NAME
)

Attempt 1: ready=READY, update=IN_PROGRESS
Attempt 2: ready=READY, update=IN_PROGRESS
Attempt 3: ready=READY, update=IN_PROGRESS
Attempt 4: ready=READY, update=IN_PROGRESS
Attempt 5: ready=READY, update=IN_PROGRESS
Attempt 6: ready=READY, update=IN_PROGRESS
Attempt 7: ready=READY, update=IN_PROGRESS
Attempt 8: ready=READY, update=IN_PROGRESS
Attempt 9: ready=READY, update=IN_PROGRESS
Attempt 10: ready=READY, update=IN_PROGRESS
Attempt 11: ready=READY, update=IN_PROGRESS
Attempt 12: ready=READY, update=IN_PROGRESS
Attempt 13: ready=READY, update=IN_PROGRESS
Attempt 14: ready=READY, update=IN_PROGRESS
Attempt 15: ready=READY, update=IN_PROGRESS
Attempt 16: ready=READY, update=IN_PROGRESS
Attempt 17: ready=READY, update=IN_PROGRESS
Attempt 18: ready=READY, update=IN_PROGRESS
Attempt 19: ready=READY, update=IN_PROGRESS
Attempt 20: ready=READY, update=IN_PROGRESS
Attempt 21: ready=READY, update=IN_PROGRESS
Attempt 22: ready=READY, update=IN_PROGRESS
Attempt 23: ready=READY, update=IN_PROGRE

In [0]:
registry_client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=VERSION_3
)

champion_v3 = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

print("Champion alias moved successfully.")
print("Champion version:", champion_v3.version)

Champion alias moved successfully.
Champion version: 3


In [0]:
final_endpoint_info = (
    deployment_client
    .get_endpoint(
        endpoint=ENDPOINT_NAME
    )
)

final_served_entity = (
    final_endpoint_info["config"]
    ["served_entities"][0]
)

final_champion = (
    registry_client
    .get_model_version_by_alias(
        name=MODEL_NAME,
        alias="Champion"
    )
)

final_versions_match = (
    str(final_champion.version)
    == str(final_served_entity["entity_version"])
)

print("FINAL PRODUCTION STATE")
print("----------------------")
print("Initial production model: Random Forest")
print("Initial version: 1")
print("Intermediate Champion: Gradient Boosting")
print("Intermediate version: 2")
print("Final production model: Decision Tree")
print("Final Champion version:", final_champion.version)
print(
    "Final endpoint version:",
    final_served_entity["entity_version"]
)
print(
    "Champion and endpoint match:",
    final_versions_match
)
print(
    "Complete automatic upgrade successful:",
    final_versions_match
    and str(final_champion.version) == str(VERSION_3)
)

FINAL PRODUCTION STATE
----------------------
Initial production model: Random Forest
Initial version: 1
Intermediate Champion: Gradient Boosting
Intermediate version: 2
Final production model: Decision Tree
Final Champion version: 3
Final endpoint version: 3
Champion and endpoint match: True
Complete automatic upgrade successful: True


In [0]:
final_model = trained_models["Decision Tree"]

final_local_predictions = final_model.predict(
    X_test
)

correct_normal_positions = np.where(
    (y_test.to_numpy() == 0)
    & (final_local_predictions == 0)
)[0]

correct_anomaly_positions = np.where(
    (y_test.to_numpy() == 1)
    & (final_local_predictions == 1)
)[0]

print(
    "Correctly predicted normal samples:",
    len(correct_normal_positions)
)

print(
    "Correctly predicted anomaly samples:",
    len(correct_anomaly_positions)
)

assert len(correct_normal_positions) > 0
assert len(correct_anomaly_positions) > 0

Correctly predicted normal samples: 52
Correctly predicted anomaly samples: 90


In [0]:
final_normal_position = int(
    correct_normal_positions[0]
)

final_normal_sample = X_test.iloc[
    [final_normal_position]
].copy()

final_normal_actual = int(
    y_test.iloc[final_normal_position]
)

final_normal_local = int(
    final_model.predict(
        final_normal_sample
    )[0]
)

final_normal_payload = create_endpoint_payload(
    final_normal_sample
)

final_normal_response = (
    deployment_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs=final_normal_payload
    )
)

final_normal_endpoint = int(
    final_normal_response["predictions"][0]
)

print("FINAL VERSION 3 — NORMAL INFERENCE")
print("----------------------------------")
print("Actual label:", final_normal_actual)
print("Local prediction:", final_normal_local)
print(
    "Endpoint prediction:",
    final_normal_endpoint
)
print(
    "Serving consistency:",
    final_normal_local
    == final_normal_endpoint
)
print(
    "Correct prediction:",
    final_normal_actual
    == final_normal_endpoint
)

FINAL VERSION 3 — NORMAL INFERENCE
----------------------------------
Actual label: 0
Local prediction: 0
Endpoint prediction: 0
Serving consistency: True
Correct prediction: True


In [0]:
final_anomaly_position = int(
    correct_anomaly_positions[0]
)

final_anomaly_sample = X_test.iloc[
    [final_anomaly_position]
].copy()

final_anomaly_actual = int(
    y_test.iloc[final_anomaly_position]
)

final_anomaly_local = int(
    final_model.predict(
        final_anomaly_sample
    )[0]
)

final_anomaly_payload = create_endpoint_payload(
    final_anomaly_sample
)

final_anomaly_response = (
    deployment_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs=final_anomaly_payload
    )
)

final_anomaly_endpoint = int(
    final_anomaly_response["predictions"][0]
)

print("FINAL VERSION 3 — ANOMALY INFERENCE")
print("-----------------------------------")
print("Actual label:", final_anomaly_actual)
print("Local prediction:", final_anomaly_local)
print(
    "Endpoint prediction:",
    final_anomaly_endpoint
)
print(
    "Serving consistency:",
    final_anomaly_local
    == final_anomaly_endpoint
)
print(
    "Correct prediction:",
    final_anomaly_actual
    == final_anomaly_endpoint
)

FINAL VERSION 3 — ANOMALY INFERENCE
-----------------------------------
Actual label: 1
Local prediction: 1
Endpoint prediction: 1
Serving consistency: True
Correct prediction: True


In [0]:
final_lifecycle = pd.DataFrame([
    {
        "Order": 1,
        "Model": "Random Forest",
        "F1 Score": get_metrics(
            "Random Forest"
        )["f1_score"],
        "Recall": get_metrics(
            "Random Forest"
        )["recall"],
        "Promotion": True,
        "Registry Version": "1",
        "Production Result": "Initial model deployed"
    },
    {
        "Order": 2,
        "Model": "XGBoost",
        "F1 Score": get_metrics(
            "XGBoost"
        )["f1_score"],
        "Recall": get_metrics(
            "XGBoost"
        )["recall"],
        "Promotion": False,
        "Registry Version": "Not registered",
        "Production Result": "Rejected by F1 gate"
    },
    {
        "Order": 3,
        "Model": "Gradient Boosting",
        "F1 Score": get_metrics(
            "Gradient Boosting"
        )["f1_score"],
        "Recall": get_metrics(
            "Gradient Boosting"
        )["recall"],
        "Promotion": True,
        "Registry Version": "2",
        "Production Result": "Automatically promoted"
    },
    {
        "Order": 4,
        "Model": "Extra Trees",
        "F1 Score": get_metrics(
            "Extra Trees"
        )["f1_score"],
        "Recall": get_metrics(
            "Extra Trees"
        )["recall"],
        "Promotion": False,
        "Registry Version": "Not registered",
        "Production Result": "Rejected by quality gate"
    },
    {
        "Order": 5,
        "Model": "Decision Tree",
        "F1 Score": get_metrics(
            "Decision Tree"
        )["f1_score"],
        "Recall": get_metrics(
            "Decision Tree"
        )["recall"],
        "Promotion": True,
        "Registry Version": "3",
        "Production Result": "Final Champion deployed"
    }
])

final_lifecycle["F1 Score"] = (
    final_lifecycle["F1 Score"].round(4)
)

final_lifecycle["Recall"] = (
    final_lifecycle["Recall"].round(4)
)

display(final_lifecycle)

Order,Model,F1 Score,Recall,Promotion,Registry Version,Production Result
1,Random Forest,0.8224,0.9167,true,1,Initial model deployed
2,XGBoost,0.8311,0.9479,false,Not registered,Rejected by F1 gate
3,Gradient Boosting,0.8505,0.9479,true,2,Automatically promoted
4,Extra Trees,0.8515,0.8958,false,Not registered,Rejected by quality gate
5,Decision Tree,0.8867,0.9375,true,3,Final Champion deployed


In [0]:
print("COMPLETE MACHINE LEARNING LIFECYCLE")
print("-----------------------------------")
print("Initial production version: 1")
print("Initial model: Random Forest")

print(
    "First successful replacement:",
    "Gradient Boosting Version 2"
)

print(
    "Final successful replacement:",
    "Decision Tree Version 3"
)

print(
    "Rejected challengers:",
    "XGBoost and Extra Trees"
)

print(
    "Final Champion version:",
    final_champion.version
)

print(
    "Final endpoint version:",
    final_served_entity[
        "entity_version"
    ]
)

print(
    "Champion and endpoint synchronized:",
    final_versions_match
)

print(
    "Normal inference correct:",
    final_normal_actual
    == final_normal_endpoint
)

print(
    "Anomaly inference correct:",
    final_anomaly_actual
    == final_anomaly_endpoint
)

complete_lifecycle_success = (
    final_versions_match
    and str(final_champion.version) == "3"
    and final_normal_actual
        == final_normal_endpoint
    and final_anomaly_actual
        == final_anomaly_endpoint
)

print(
    "Complete lifecycle successful:",
    complete_lifecycle_success
)

COMPLETE MACHINE LEARNING LIFECYCLE
-----------------------------------
Initial production version: 1
Initial model: Random Forest
First successful replacement: Gradient Boosting Version 2
Final successful replacement: Decision Tree Version 3
Rejected challengers: XGBoost and Extra Trees
Final Champion version: 3
Final endpoint version: 3
Champion and endpoint synchronized: True
Normal inference correct: True
Anomaly inference correct: True
Complete lifecycle successful: True


## Final Conclusion

The project successfully demonstrates a complete production-oriented machine
learning lifecycle for CNC milling anomaly detection.

Random Forest was initially registered and deployed as Version 1. XGBoost was
evaluated but rejected because it did not meet the minimum F1-score improvement
requirement.

Gradient Boosting passed the automatic quality gate and replaced Random Forest
as Version 2. Extra Trees was subsequently rejected because its performance did
not satisfy the combined F1-score and anomaly-recall requirements.

The final Decision Tree challenger achieved the strongest F1-score while
maintaining acceptable anomaly recall. It was automatically registered as
Version 3, assigned the Champion alias and deployed through the existing
Databricks serving endpoint.

The final endpoint was validated using both normal and anomalous machining
segments, and its predictions matched the locally evaluated Champion model.